In [22]:
import pandas as pd
import numpy as np

In [23]:
#load CSVs from "data"
stock_data = pd.read_csv("../data/stock_data.csv", index_col=0, parse_dates=True)
benchmark = pd.read_csv("../data/benchmark.csv", index_col=0, parse_dates=True)
meta_df = pd.read_csv("../data/metadata.csv", index_col=0)

In [25]:
#calculate & save daily returns in a new CSV
returns_df         = stock_data.pct_change().dropna()
cumulative_returns = (1 + returns_df).cumprod() * 100
cumulative_returns.to_csv("../data/cumulative_returns.csv")

In [30]:
#calculate benchmark returns (from S&P 500)
benchmark_returns = benchmark["^GSPC"].pct_change().dropna()

#sync date inidces to make it accessible for numpy
returns_aligned, benchmark_aligned = returns_df.align(benchmark_returns, join="inner", axis=0)

#calculate alpha
alpha_results = {}
for ticker in ["AAPL", "MSFT", "JPM"]:
    _, alpha = np.polyfit(benchmark_aligned, returns_aligned[ticker], 1)
    alpha_results[ticker] = {"alpha_annualized": alpha * 252}

alpha_df = pd.DataFrame(alpha_results).T

#save in CSV
alpha_df.to_csv("../data/alpha_metrics.csv")

In [31]:
#calculate volatility (p.a.)
volatility = returns_df.std() * np.sqrt(252)

#calculate volatility (30d)
rolling_volatility = returns_df.rolling(window=30).std() * np.sqrt(252)

#safe in CSVs
volatility.to_csv("../data/volatility.csv")
rolling_volatility.to_csv("../data/rolling_volatility.csv")

In [32]:
#calculate sharp ratio
risk_free_rate = 0.05  # US Treasury 5%

annual_returns = returns_df.mean() * 252
sharpe_ratio   = (annual_returns - risk_free_rate) / volatility

#save in CSV
sharpe_ratio.to_csv("../data/sharpe_ratio.csv")


✅ returns.csv gespeichert!
✅ risk_metrics.csv gespeichert!
